# NC-SSM Checkpoint Download
Google Drive에서 모든 체크포인트를 다운로드합니다.

In [ ]:
from google.colab import drive, files
import os, zipfile

drive.mount('/content/drive')

# 검색할 Drive 경로들
DRIVE_DIRS = [
    '/content/drive/MyDrive/NC-SSM-TASLP/checkpoints_full',
    '/content/drive/MyDrive/NanoMamba/checkpoints_full',
    '/content/drive/MyDrive/NanoMamba-TASLP/checkpoints_full',
    '/content/drive/MyDrive/Colab Notebooks',
]

# 특히 찾아야 할 모델들
WANTED = [
    'NanoMamba-NC-Matched',      # NC-SSM (7.4K)
    'NanoMamba-NC-Large',        # NC-SSM-Large (10.2K)
    'NanoMamba-NC-Large-NASG',   # NC-SSM-Large+NASG
    'NanoMamba-NC-15K',          # NC-SSM-15K
    'NanoMamba-NC-20K',          # NC-SSM-20K
    'NanoMamba-NC-15K-SS',       # NC-SSM-15K weight sharing
    'NanoMamba-NC-20K-SS',       # NC-SSM-20K weight sharing
    'NanoMamba-NC-NanoSE',       # NC-SSM + NanoSE
    'BC-ResNet-1',               # CNN baseline
    'DS-CNN-S',                  # Large CNN
    'NanoMamba-Matched-DualPCEN-v2-SSMv2',  # SSM baseline
    'NanoMamba-Matched-SM',      # SM-SSM ablation
    'SAGN',                      # SAGN ablation
]

# 모든 Drive 경로에서 best.pt 검색
models_found = {}
for drive_dir in DRIVE_DIRS:
    if not os.path.exists(drive_dir):
        continue
    for root, dirs, fnames in os.walk(drive_dir):
        for f in fnames:
            if f == 'best.pt':
                full = os.path.join(root, f)
                name = os.path.basename(os.path.dirname(full))
                if name not in models_found:
                    size_kb = os.path.getsize(full) / 1024
                    models_found[name] = (full, size_kb)

print('=' * 60)
print('FOUND CHECKPOINTS:')
print('=' * 60)
for name in sorted(models_found):
    path, size = models_found[name]
    tag = ' ★' if name in WANTED else ''
    print(f'  ✓ {name:45s} ({size:.0f} KB){tag}')

print(f'\nTotal: {len(models_found)} checkpoints')
missing = [w for w in WANTED if w not in models_found]
if missing:
    print(f'\nMISSING ({len(missing)}): {", ".join(missing)}')

In [ ]:
# ZIP으로 묶어서 한번에 다운로드
ZIP_PATH = '/content/all_checkpoints.zip'

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in sorted(models_found):
        path, size = models_found[name]
        zf.write(path, f'{name}/best.pt')
        print(f'  packed: {name}/best.pt ({size:.0f} KB)')

zip_size = os.path.getsize(ZIP_PATH) / 1024 / 1024
print(f'\nZIP: {zip_size:.1f} MB → Downloading...')
files.download(ZIP_PATH)